# Backpropagation from scratch

Build a two-layer NumPy network for XOR, derive its gradients, and compare them with finite differences before training. This is a mechanics lesson using four examples, not a held-out performance benchmark.

**Prerequisites:** matrix multiplication, derivatives, and sigmoid. **Environment:** base requirements. **Execution:** restart the kernel and run all cells. No dataset files, downloads, or GPU are required.


In [ ]:
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
X = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = np.array([[0.], [1.], [1.], [0.]])
parameters = {'W1': rng.normal(0, 0.6, (2, 8)), 'b1': np.zeros((1, 8)),
              'W2': rng.normal(0, 0.6, (8, 1)), 'b2': np.zeros((1, 1))}
print({name: value.shape for name, value in parameters.items()})


## Forward pass and binary cross-entropy

For a batch $X$, the hidden activation is $H=\tanh(XW_1+b_1)$ and the output logit is $Z=HW_2+b_2$. Binary cross-entropy from logits is $\operatorname{mean}(\log(1+e^Z)-yZ)$. `logaddexp` computes this without directly exponentiating large positive logits.

The output derivative is $(\sigma(Z)-y)/N$. Chain it through $W_2$ and $\tanh'(a)=1-H^2$ to compute the first-layer derivatives. Bias gradients sum across the batch.


In [ ]:
def forward(parameters, features):
    hidden = np.tanh(features @ parameters['W1'] + parameters['b1'])
    logits = hidden @ parameters['W2'] + parameters['b2']
    probabilities = np.exp(-np.logaddexp(0, -logits))
    return hidden, logits, probabilities

def loss_and_gradients(parameters, features, targets):
    hidden, logits, probabilities = forward(parameters, features)
    loss = np.mean(np.logaddexp(0, logits) - targets * logits)
    output_delta = (probabilities - targets) / len(features)
    hidden_delta = (output_delta @ parameters['W2'].T) * (1 - hidden ** 2)
    gradients = {'W1': features.T @ hidden_delta,
                 'b1': hidden_delta.sum(axis=0, keepdims=True),
                 'W2': hidden.T @ output_delta,
                 'b2': output_delta.sum(axis=0, keepdims=True)}
    return float(loss), gradients


## Check every gradient numerically

Perturb one parameter at a time by $\epsilon$ and estimate its derivative as $[L(\theta+\epsilon)-L(\theta-\epsilon)]/(2\epsilon)$. The numerical calculation uses the loss only, independently of the backpropagation formulas. Small floating-point errors are expected; a large discrepancy would indicate an incorrect derivative.


In [ ]:
_, analytic = loss_and_gradients(parameters, X, y)
numerical = {name: np.zeros_like(value) for name, value in parameters.items()}
epsilon = 1e-6
for name, value in parameters.items():
    for index in np.ndindex(value.shape):
        plus, minus = copy.deepcopy(parameters), copy.deepcopy(parameters)
        plus[name][index] += epsilon
        minus[name][index] -= epsilon
        loss_plus, _ = loss_and_gradients(plus, X, y)
        loss_minus, _ = loss_and_gradients(minus, X, y)
        numerical[name][index] = (loss_plus - loss_minus) / (2 * epsilon)
analytic_vector = np.concatenate([value.ravel() for value in analytic.values()])
numerical_vector = np.concatenate([value.ravel() for value in numerical.values()])
relative_error = np.linalg.norm(analytic_vector - numerical_vector) / (
    np.linalg.norm(analytic_vector) + np.linalg.norm(numerical_vector) + 1e-12
)
assert relative_error < 1e-6, relative_error
print('Checked parameter count:', len(analytic_vector))
print('Gradient relative error:', relative_error)


## Train with gradient descent

The learning rate scales the update. We deliberately train on the full truth table and display that same table to illustrate learning a nonlinear mapping. These four training predictions do not estimate generalization.


In [ ]:
losses = []
learning_rate = 1.0
for epoch in range(1500):
    loss, gradients = loss_and_gradients(parameters, X, y)
    losses.append(loss)
    for name in parameters:
        parameters[name] -= learning_rate * gradients[name]
_, _, probabilities = forward(parameters, X)
display(pd.DataFrame({'x1': X[:, 0], 'x2': X[:, 1], 'target': y.ravel(),
                      'probability': probabilities.ravel(),
                      'prediction': (probabilities.ravel() >= 0.5).astype(int)}))
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(losses)
ax.set(xlabel='Update', ylabel='Binary cross-entropy', title='XOR training loss')
plt.show()
assert np.isfinite(losses).all()


## Practice

Try two hidden units or a smaller learning rate. Replace `tanh` with another activation and re-derive its gradient; run the finite-difference check again before training.

**References:** [NumPy logaddexp](https://numpy.org/doc/stable/reference/generated/numpy.logaddexp.html), [PyTorch autograd introduction](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html).
